In [ ]:
# ============================================================
# KG1 v51 PERFECT — SINGLE CELL (tudo em um)
# Config v30 (scored 0.68) + 9500 solver-enhanced CoTs (100% ACC)
# Score esperado: 0.74-0.80
# ============================================================

# ==================== INSTALL ====================
!pip install -q peft datasets accelerate trl huggingface_hub safetensors pandas

import subprocess, sys, os, json, random, time, zipfile, shutil, re, math, types
from datetime import datetime, timezone
from collections import Counter

# ==================== MAMBA-SSM + CAUSAL_CONV1D STUB ====================
# Blackwell (sm_120) nao compila mamba-ssm. Injetar stubs.
class _Stub:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw): return a[0] if a else None
    def __getattr__(self, name): return _Stub()

try:
    import mamba_ssm
    print(f'mamba-ssm {mamba_ssm.__version__} OK')
except ImportError:
    print('mamba-ssm not available, injecting stubs...')
    mamba_ssm = types.ModuleType('mamba_ssm')
    mamba_ssm.__version__ = '0.0.0'
    for submod in ['ops', 'ops.triton', 'ops.triton.layernorm_gated',
                   'ops.triton.selective_state_update', 'ops.triton.ssd_combined',
                   'utils', 'utils.generation']:
        m = types.ModuleType(f'mamba_ssm.{submod}')
        for attr in ['RMSNormGated','rmsnorm_fn','selective_state_update',
                     'mamba_chunk_scan_combined','mamba_split_conv1d_scan_combined',
                     'InferenceParams','GenerationMixin']:
            setattr(m, attr, _Stub)
        sys.modules[f'mamba_ssm.{submod}'] = m
        parts = submod.split('.')
        parent = mamba_ssm
        for p in parts[:-1]:
            parent = getattr(parent, p, types.ModuleType(p))
        setattr(parent, parts[-1], m)
    sys.modules['mamba_ssm'] = mamba_ssm
    print('mamba-ssm STUB injected')

try:
    import causal_conv1d
    print(f'causal_conv1d OK')
except ImportError:
    cc1d = types.ModuleType('causal_conv1d')
    cc1d.causal_conv1d_fn = _Stub()
    cc1d.causal_conv1d_update = _Stub()
    cc1d.__spec__ = types.ModuleType('_spec')
    cc1d.__spec__.name = 'causal_conv1d'
    cc1d.__spec__.origin = 'stub'
    sys.modules['causal_conv1d'] = cc1d
    sys.modules['causal_conv1d.causal_conv1d_interface'] = cc1d
    print('causal_conv1d STUB injected')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')

# ==================== MONKEY-PATCH ====================
try:
    from transformers.utils.import_utils import is_flash_attn_greater_or_equal_2_10
except ImportError:
    import transformers.utils.import_utils as _tiu
    _tiu.is_flash_attn_greater_or_equal_2_10 = lambda: False
    print('Patched: is_flash_attn_greater_or_equal_2_10')

import pandas as pd
from huggingface_hub import HfApi, login, hf_hub_download

# ==================== AUTH ====================
def _get_secret(*names):
    try:
        from google.colab import userdata
        for n in names:
            v = userdata.get(n)
            if v: return v
    except Exception: pass
    for n in names:
        v = os.environ.get(n)
        if v: return v
    return ''

HF_TOKEN = _get_secret('HF_KEY', 'HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF login OK')

KAGGLE_USERNAME = _get_secret('KAGGLE_USERNAME') or 'felipe1983'
KAGGLE_KEY = _get_secret('KAGGLE_KEY')
if KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    kpath = os.path.expanduser('~/.kaggle/kaggle.json')
    with open(kpath, 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(kpath, 0o600)
    print(f'Kaggle: {KAGGLE_USERNAME}')

api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()

# ==================== CONFIG ====================
DATA_REPO = 'felipesp1983/kg1-nemotron-training'
OUTPUT_REPO = 'felipesp1983/kg1-nemotron-lora-v51-perfect'
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
COMPETITION = 'nvidia-nemotron-model-reasoning-challenge'

N_EXAMPLES = 5000
N_EPOCHS = 2
SUBMIT_STEPS = [200, 400, 600, 800, 1000, 1200]

CONFIG = {
    'lora_rank': 32, 'lora_alpha': 16, 'lora_dropout': 0.05,
    'target_modules': 'all-linear', 'learning_rate': 5e-5,
    'per_device_batch_size': 1, 'gradient_accumulation_steps': 8,
    'max_length': 1024, 'warmup_ratio': 0.05, 'weight_decay': 0.01,
    'lr_scheduler': 'cosine', 'optim': 'adamw_torch',
    'output_dir': '/tmp/kg1_output/v51',
}

# Triton patch
try:
    for ptxas_src in ['/usr/local/cuda-12.8/bin/ptxas', '/usr/local/cuda/bin/ptxas']:
        if os.path.exists(ptxas_src):
            target = os.path.join(os.path.dirname(shutil.which('python') or '/usr/bin/python'), 'ptxas')
            if not os.path.exists(target):
                shutil.copy2(ptxas_src, target)
                print(f'Triton ptxas patched: {target}')
            break
except Exception: pass

print(f'Config: {N_EXAMPLES}ex, {N_EPOCHS}ep, LR={CONFIG["learning_rate"]}, r={CONFIG["lora_rank"]}, a={CONFIG["lora_alpha"]}')

# ==================== LOAD DATA ====================
print('\n=== Loading v51 PERFECT data ===')
all_examples = []
data_loaded = False

try:
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                    filename='data/sft_v51_perfect.jsonl', local_dir='/tmp/kg1_data')
    with open('/tmp/kg1_data/data/sft_v51_perfect.jsonl') as f:
        for line in f:
            all_examples.append(json.loads(line))
    data_loaded = True
    print(f'Loaded from HF: {len(all_examples)} examples')
except Exception as e:
    print(f'HF failed: {e}')

if not data_loaded:
    print('Falling back to train.csv...')
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                    filename='data/train.csv', local_dir='/tmp/kg1_data')
    train_df = pd.read_csv('/tmp/kg1_data/data/train.csv')
    for _, row in train_df.iterrows():
        all_examples.append({
            'prompt': row['prompt'] + '\\nPut your final answer inside \\\\boxed{}.',
            'completion': f'\\\\boxed{{{row["answer"]}}}',
            'family': 'unknown',
        })
    print(f'Loaded {len(all_examples)} raw examples (fallback)')

def classify(text):
    p = text.lower()
    if 'bit manipulation' in p: return 'bit'
    if 'gravitational' in p: return 'grav'
    if 'unit conversion' in p or 'measurement' in p: return 'unit'
    if 'numeral' in p: return 'num'
    if 'encryption' in p: return 'enc'
    if 'transformation' in p: return 'eq'
    return 'other'

# Stratified sampling
print(f'\n=== Sampling {N_EXAMPLES} examples ===')
random.seed(42)
by_family = {}
for ex in all_examples:
    fam = ex.get('family') or classify(ex.get('prompt', ''))
    by_family.setdefault(fam, []).append(ex)

shares = {'grav':1, 'unit':1, 'num':1, 'enc':1, 'cipher':1, 'bit':1.5,
          'eq':2.5, 'equation':2.5, 'gravity':1, 'numeral':1}
total_shares = sum(shares.get(f, 1.0) for f in by_family)
base_n = N_EXAMPLES / total_shares

examples = []
for fam, pool in by_family.items():
    n_want = int(base_n * shares.get(fam, 1.0))
    if n_want <= len(pool): selected = random.sample(pool, n_want)
    else: selected = pool + random.choices(pool, k=n_want - len(pool))
    examples.extend(selected)
random.shuffle(examples)
examples = examples[:N_EXAMPLES]

formatted = [{'messages': [{'role':'user','content':ex.get('prompt','')},
              {'role':'assistant','content':ex.get('completion','')}]} for ex in examples]

fam_counts = Counter(classify(e['messages'][0]['content']) for e in formatted)
print(f'Dataset: {len(formatted)} examples')
for fam, cnt in sorted(fam_counts.items()):
    print(f'  {fam}: {cnt} ({cnt/len(formatted)*100:.1f}%)')

# ==================== DOWNLOAD MODEL (sequential) ====================
print('\n=== Downloading model ===')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
from huggingface_hub import snapshot_download
model_path = snapshot_download(MODEL_NAME, local_dir='/tmp/nemotron_model', max_workers=1)
print(f'Model downloaded to: {model_path}')

# ==================== LOAD MODEL ====================
print('\n=== Loading model (BF16) ===')
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

_gpu_cap = float(f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}')
IS_BLACKWELL = _gpu_cap >= 10.0

_model_kwargs = dict(device_map={'': 0}, trust_remote_code=True, torch_dtype=torch.bfloat16)

if IS_BLACKWELL:
    _cfg = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
    if hasattr(_cfg, 'use_mamba_kernels'):
        _cfg.use_mamba_kernels = False
        _model_kwargs['config'] = _cfg
        print(f'[BLACKWELL sm_{int(_gpu_cap*10)}] Mamba kernels DISABLED')

model = AutoModelForCausalLM.from_pretrained(model_path, **_model_kwargs)

fp_count = 0
for module in model.modules():
    if hasattr(module, 'is_fast_path_available'):
        module.is_fast_path_available = False
        fp_count += 1
print(f'Model: {model.num_parameters()/1e9:.1f}B params, fast_path disabled ({fp_count})')

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer: vocab={tokenizer.vocab_size}')

# ==================== APPLY LORA ====================
print(f'\n=== Applying LoRA (r={CONFIG["lora_rank"]}, alpha={CONFIG["lora_alpha"]}) ===')
from peft import LoraConfig, get_peft_model

model.enable_input_require_grads()
lora_config = LoraConfig(
    r=CONFIG['lora_rank'], lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'], target_modules=CONFIG['target_modules'],
    bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ==================== PREPARE DATASET ====================
print('\n=== Preparing dataset ===')
from datasets import Dataset

texts = [tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False) for ex in formatted]
ds = Dataset.from_dict({'text': texts})
print(f'Dataset: {len(ds)} examples')
sample_lens = [len(tokenizer(t)['input_ids']) for t in texts[:100]]
print(f'Tokens (100 sample): min={min(sample_lens)}, max={max(sample_lens)}, mean={sum(sample_lens)/len(sample_lens):.0f}')

# ==================== TRAIN ====================
print('\n=== Training v51 PERFECT ===')
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

os.makedirs(CONFIG['output_dir'], exist_ok=True)

training_args = SFTConfig(
    output_dir=CONFIG['output_dir'], dataset_text_field='text',
    max_length=CONFIG['max_length'], packing=False,
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'], warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'], lr_scheduler_type=CONFIG['lr_scheduler'],
    optim=CONFIG['optim'], bf16=True, logging_steps=5,
    save_strategy='steps', save_steps=100, save_total_limit=15,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none', dataloader_num_workers=0, max_grad_norm=1.0,
)

class AutoUploadCallback(TrainerCallback):
    def __init__(self, repo_id, submit_steps):
        self.repo_id = repo_id
        self.submit_steps = set(submit_steps)
        self.submitted = set()
        self.hf_api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()
        try: self.hf_api.create_repo(repo_id, private=True, exist_ok=True)
        except: pass

    def on_save(self, args, state, control, **kwargs):
        import glob as g
        step = state.global_step
        loss_val = 'N/A'
        if state.log_history:
            for entry in reversed(state.log_history):
                if 'loss' in entry: loss_val = entry['loss']; break
        ckpts = sorted(g.glob(f'{args.output_dir}/checkpoint-*'))
        if not ckpts: return
        try:
            self.hf_api.upload_folder(folder_path=ckpts[-1], path_in_repo=f'checkpoint-{step}',
                repo_id=self.repo_id, commit_message=f'Step {step} | Loss {loss_val}')
            print(f'\n>>> HF upload OK: step {step}, loss={loss_val}')
        except Exception as e:
            print(f'\n>>> HF upload FAILED: {e}')
        if step in self.submit_steps and step not in self.submitted:
            print(f'  Step {step} ready for submit')
            self.submitted.add(step)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        loss = logs.get('loss', 0)
        if isinstance(loss, float) and (math.isnan(loss) or math.isinf(loss)):
            print(f'\n!!! CRITICAL: Loss NaN/Inf at step {state.global_step}')
            control.should_training_stop = True
            return
        if isinstance(loss, (int, float)) and loss > 30.0 and state.global_step > 5:
            print(f'\n!!! Loss explosion {loss:.2f} at step {state.global_step}')
            control.should_training_stop = True

trainer = SFTTrainer(
    model=model, train_dataset=ds, processing_class=tokenizer,
    args=training_args,
    callbacks=[AutoUploadCallback(OUTPUT_REPO, SUBMIT_STEPS)],
)

total_steps = (len(ds) // CONFIG['gradient_accumulation_steps']) * N_EPOCHS
print(f'Steps: ~{total_steps}, Est time: ~{total_steps * 55 / 3600:.1f}h')
print(f'Submit steps: {sorted(SUBMIT_STEPS)}')

start = time.time()
try:
    trainer.train()
except Exception as e:
    print(f'\n!!! Training error: {e}')
    try:
        model.save_pretrained(CONFIG['output_dir'])
        api.upload_folder(folder_path=CONFIG['output_dir'], repo_id=OUTPUT_REPO,
                         path_in_repo='emergency', commit_message=f'Emergency: {str(e)[:80]}')
    except: pass

elapsed = time.time() - start
print(f'\nTraining complete: {elapsed/3600:.2f}h')

# ==================== SAVE FINAL ====================
print('\n=== Saving final adapter ===')
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])

final_loss = 'N/A'
if trainer.state.log_history:
    for entry in reversed(trainer.state.log_history):
        if 'loss' in entry: final_loss = entry['loss']; break

try:
    api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=CONFIG['output_dir'], repo_id=OUTPUT_REPO,
        commit_message=f'v51 FINAL: {len(formatted)}ex, {N_EPOCHS}ep, loss={final_loss}')
    print(f'Uploaded: https://huggingface.co/{OUTPUT_REPO}')
except Exception as e:
    print(f'Upload failed: {e}')

# ==================== SMART STRIP + SUBMIT ====================
print('\n=== Smart Strip (step 400) ===')
import glob
from safetensors.torch import load_file, save_file

ckpt_dir = f'{CONFIG["output_dir"]}/checkpoint-400'
if not os.path.exists(ckpt_dir):
    ckpts = sorted(glob.glob(f'{CONFIG["output_dir"]}/checkpoint-*'), key=lambda x: int(x.split('-')[-1]))
    ckpt_dir = ckpts[-1] if ckpts else CONFIG['output_dir']
    print(f'Using: {ckpt_dir}')

sf_path = os.path.join(ckpt_dir, 'adapter_model.safetensors')
cfg_path = os.path.join(ckpt_dir, 'adapter_config.json')

tensors = load_file(sf_path)
routed_re = re.compile(r'\.experts\.\d+\.')
keep = {k: v for k, v in tensors.items() if not routed_re.search(k)}
removed = len(tensors) - len(keep)
print(f'Kept: {len(keep)} | Removed routed experts: {removed}')

out_dir = '/tmp/kg1_submit/stripped'
os.makedirs(out_dir, exist_ok=True)
save_file(keep, os.path.join(out_dir, 'adapter_model.safetensors'))

with open(cfg_path) as f: cfg = json.load(f)
kept_mods = set()
for k in keep:
    for m in ['q_proj','k_proj','v_proj','o_proj','in_proj','out_proj','up_proj','down_proj','gate']:
        if m in k: kept_mods.add(m)
cfg['target_modules'] = sorted(kept_mods)
with open(os.path.join(out_dir, 'adapter_config.json'), 'w') as f:
    json.dump(cfg, f, indent=2)

zip_path = '/tmp/kg1_submit/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['adapter_config.json', 'adapter_model.safetensors']:
        zf.write(os.path.join(out_dir, fname), fname)
print(f'ZIP: {os.path.getsize(zip_path)/1e6:.1f} MB')

# Submit
step_str = ckpt_dir.split('-')[-1] if 'checkpoint' in ckpt_dir else 'final'
desc = f'v51-perfect-step{step_str}-smart-strip'
!kaggle competitions submit -c {COMPETITION} -f {zip_path} -m "{desc}"

print(f'\n{"="*60}')
print(f'  v51 PERFECT COMPLETE')
print(f'  Examples: {len(formatted)} | Epochs: {N_EPOCHS}')
print(f'  Final loss: {final_loss}')
print(f'  Time: {elapsed/3600:.2f}h')
print(f'  Submitted: {desc}')
print(f'{"="*60}')
